In [1]:
import os, sys, pathlib, importlib

In [ ]:
from PIL import Image
import torch
from transformers import CLIPProcessor, CLIPModel
from ultralytics import YOLO

# ----- 1. MODELS -----
# YOLOv8 人物检测
yolo_model = YOLO("yolov8n.pt")  # 或 yolov8n-seg 根据需要，n是轻量模型

# CLIP 文本 + 图像
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

## 标签文本
labels = [
        "Portrait professionnel : personne occupe la majorité de l'image, fond simple ou neutre",
        "Photo de vie : personne dans un contexte quotidien, fond riche, personne plus petite dans l'image"
    ]



# ----- 2. DATA -----
# images = ["avatar1.jpg", "avatar2.jpg", "avatar3.jpg"]  # 小批量测试
IMG_FOLDER="../images\host_pic"
images=[]

## marche le mieux quand on fournit sys et user prompt à la fois
for filename in os.listdir(IMG_FOLDER):
    if not filename.endswith((".jpg",".JPG",".jpeg",".png","tif")):
        continue #==skip
    img_path=os.path.join(IMG_FOLDER, filename)
    images.append(img_path)
    
    
    

# ----- 3. Pipeline -----
results = []
for img_path in images:
    image = Image.open(img_path).convert("RGB")
    
    # YOLO 人物检测
    yolo_preds = yolo_model(image)
    # 判断是否检测到人物（YOLO返回框数量）
    if len(yolo_preds[0].boxes) == 0:
        results.append({
            "image": img_path,
            "label": "no_person",
            "confidence": 1.0,
            "bbox": None
        })
        continue
    
    # CLIP 分类
    # 标签更新
    

    # CLIP 分类同之前代码
    inputs = clip_processor(text=labels, images=image, return_tensors="pt", padding=True)
    outputs = clip_model(**inputs)
    probs = outputs.logits_per_image.softmax(dim=1)
    conf, idx = probs.max(dim=1)
    pred_label = ["pro_style", "life_style"][idx.item()]  # 对应原始分类标签

    # inputs = clip_processor(text=labels, images=image, return_tensors="pt", padding=True)
    # outputs = clip_model(**inputs)
    # probs = outputs.logits_per_image.softmax(dim=1)
    # conf, idx = probs.max(dim=1)
    
    results.append({
        "image": img_path,
        "host_id":os.path.splitext(os.path.basename(img_path))[0],
        # "label": labels[idx.item()],
        "label":pred_label,
        "confidence": conf.item(),
        "bbox": [box.xyxy.tolist() for box in yolo_preds[0].boxes]  # YOLO检测框
    })

# 输出示例
import json
print(json.dumps(results, indent=2))


0: 640x640 1 person, 49.4ms
Speed: 17.6ms preprocess, 49.4ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 48.8ms
Speed: 2.5ms preprocess, 48.8ms inference, 0.4ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 person, 41.9ms
Speed: 2.6ms preprocess, 41.9ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 person, 51.2ms
Speed: 3.4ms preprocess, 51.2ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 dog, 1 horse, 43.5ms
Speed: 4.1ms preprocess, 43.5ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 42.3ms
Speed: 2.3ms preprocess, 42.3ms inference, 0.5ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 3 persons, 42.1ms
Speed: 3.5ms preprocess, 42.1ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 1 person, 1 tie, 48.6ms
Speed: 2.4ms preprocess, 48.6ms inference, 0.7ms postprocess

In [13]:
with open("../images\host_pic/results_clf.json",'w', encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
    
    
# results

In [3]:
import json
with open("../images\host_pic/results_clf.json","r", encoding="utf-8")as f:
    results=json.load(f)
print(results)

[{'image': '../images\\host_pic\\102571900.jpg', 'host_id': '102571900', 'label': 'pro_style', 'confidence': 0.7503951787948608, 'bbox': [[[25.913850784301758, 1.4830470085144043, 213.36671447753906, 224.19583129882812]]]}, {'image': '../images\\host_pic\\106294215.jpg', 'label': 'no_person', 'confidence': 1.0, 'bbox': None}, {'image': '../images\\host_pic\\106365215.jpg', 'host_id': '106365215', 'label': 'life_style', 'confidence': 0.6560038924217224, 'bbox': [[[80.45903778076172, 62.655948638916016, 200.13754272460938, 224.81069946289062]]]}, {'image': '../images\\host_pic\\137154154.jpg', 'host_id': '137154154', 'label': 'life_style', 'confidence': 0.7466520071029663, 'bbox': [[[26.576839447021484, 98.66331481933594, 150.2098388671875, 224.64576721191406]]]}, {'image': '../images\\host_pic\\212791574.jpg', 'host_id': '212791574', 'label': 'life_style', 'confidence': 0.5426696538925171, 'bbox': [[[12.427897453308105, 1.5488362312316895, 219.47572326660156, 222.5258026123047]], [[7.67

In [ ]:
import json
with open("../images\host_pic/metadata_host_pic.json","r", encoding="utf-8")as f:
    metadata=json.load(f)
with open("../images\host_pic/results_clf.json","r", encoding="utf-8")as f:
    results=json.load(f)


host_data=[]
for  host in metadata:
    host_id=host["host_id"]
    
    pred_label=[res for res in results if res["host_id"]=host_id ]
    true_label=metadata["true_label"]
    
    print(host["host_id"], host["true_label"],"\n")


102571900 pro_style 

106294215 life_style 

106365215 life_style 

137154154 life_style 

212791574 no_person 

2379345 no_person 

24654560 life_style 

2798386 pro_style 

28470251 pro_style 

32741638 pro_style 

336591839 no_person 

425502119 no_person 

517697918 life_style 

52438163 life_style 

52801103 no_person 

553099349 life_style 

57226046 life_style 

71320446 pro_style 

873444 life_style 

88933385 no_person 

